In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# --- Project root (robust to folder moves) ---
# Default to the known location; fall back to the current working directory
# if the project has been moved again (launch JupyterLab from the project folder).
PROJECT_ROOT = Path.home() / "Claude" / "Projects" / "Directed Study" / "Ransomware"
if not (PROJECT_ROOT / "features").exists():
    PROJECT_ROOT = Path.cwd()
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.manifold import TSNE
import umap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
DATA_ROOT    = PROJECT_ROOT / "dataset"
FEATURES_DIR = PROJECT_ROOT / "features"

# Class definitions
RANSOMWARE_CLASSES    = ["WannaCry", "Ryuk", "REvil", "LockBit", "Darkside", "Conti"]
LEGIT_ENC_CLASSES     = ["AESCrypt", "SDelete", "Zip"]
BENIGN_CLASSES        = ["Firefox", "Office", "Idle"]
ALL_ORIGINAL_CLASSES  = RANSOMWARE_CLASSES + LEGIT_ENC_CLASSES + BENIGN_CLASSES
VARIANT_CLASSES       = [f"Conti_{i:02d}" for i in range(1, 8)]

# 3-way label mapping
def get_3way_label(class_name):
    if class_name in RANSOMWARE_CLASSES: return "ransomware"
    if class_name in LEGIT_ENC_CLASSES:  return "legit_encryption"
    if class_name in BENIGN_CLASSES:     return "benign"
    return "unknown"

# Colors
CLASS_COLORS = {
    "ransomware":       "#e6194b",
    "legit_encryption": "#f58231",
    "benign":           "#4363d8",
    "variant":          "#3cb44b",
}

# T_window parameters
T_WINDOW    = 0.1
T_D         = 30.0
T_WINDOW_NS = int(T_WINDOW * 1_000_000_000)
T_D_NS      = int(T_D      * 1_000_000_000)

print("Config loaded.")
print(f"3-way classes:")
print(f"  Ransomware:        {RANSOMWARE_CLASSES}")
print(f"  Legit Encryption:  {LEGIT_ENC_CLASSES}")
print(f"  Benign:            {BENIGN_CLASSES}")
print(f"  Zero-day proxies:  {VARIANT_CLASSES}")

/home/arshad/Claude/Projects/Directed Study/Ransomware/ransmap_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Config loaded.
3-way classes:
  Ransomware:        ['WannaCry', 'Ryuk', 'REvil', 'LockBit', 'Darkside', 'Conti']
  Legit Encryption:  ['AESCrypt', 'SDelete', 'Zip']
  Benign:            ['Firefox', 'Office', 'Idle']
  Zero-day proxies:  ['Conti_01', 'Conti_02', 'Conti_03', 'Conti_04', 'Conti_05', 'Conti_06', 'Conti_07']


In [2]:
def log_lines_to_text(ata_write_lines, mem_write_lines):
    """
    Convert raw log lines from a 0.1s window into a natural language
    paragraph describing the behavioral pattern.

    Preserves individual access events — not statistics.
    Each line becomes a token describing one actual hardware event.
    Empty windows get a meaningful null description.
    """
    tokens = []

    # Storage write events
    if ata_write_lines.empty:
        tokens.append("no storage write activity")
    else:
        for _, row in ata_write_lines.iterrows():
            size    = int(row.iloc[3]) if not pd.isna(row.iloc[3]) else 0
            entropy = float(row.iloc[4]) if not pd.isna(row.iloc[4]) else -1.0

            if entropy < 0:
                ent_label = "unknown"
            elif entropy > 0.9:
                ent_label = "very-high-entropy"
            elif entropy > 0.7:
                ent_label = "high-entropy"
            elif entropy > 0.4:
                ent_label = "moderate-entropy"
            elif entropy > 0.1:
                ent_label = "low-entropy"
            else:
                ent_label = "near-zero-entropy"

            size_label = "large-write" if size > 65536 else \
                         "medium-write" if size > 4096 else "small-write"

            tokens.append(f"storage-write {size_label} {ent_label}")

    # Memory write events
    if mem_write_lines.empty:
        tokens.append("no memory write activity")
    else:
        for _, row in mem_write_lines.iterrows():
            entropy  = float(row.iloc[4]) if not pd.isna(row.iloc[4]) else 0.0
            mem_type = int(row.iloc[5])   if not pd.isna(row.iloc[5]) else 1

            type_label = {1: "4kb-page", 2: "2mb-page", 4: "mmio"}.get(mem_type, "unknown-page")

            if entropy > 0.9:   ent_label = "very-high-entropy"
            elif entropy > 0.7: ent_label = "high-entropy"
            elif entropy > 0.4: ent_label = "moderate-entropy"
            elif entropy > 0.1: ent_label = "low-entropy"
            else:               ent_label = "near-zero-entropy"

            tokens.append(f"memory-write {type_label} {ent_label}")

    # Truncate to fit within 512 token budget (73 events max)
    tokens = tokens[:73]
    return " . ".join(tokens)


# Test on a real execution
test_dir = DATA_ROOT / "ddr4-2133-16g" / "WannaCry"
test_exec = sorted([d for d in test_dir.iterdir() if d.is_dir()])[0]

ata_w = pd.read_csv(test_exec / "ata_write.csv", header=None)
mem_w = pd.read_csv(test_exec / "mem_write.csv", header=None)

# Get first window
ref_ns = min(
    int(ata_w.iloc[0, 0]) * 1_000_000_000 + int(ata_w.iloc[0, 1]),
    int(mem_w.iloc[0, 0]) * 1_000_000_000 + int(mem_w.iloc[0, 1])
)

ata_rel = (ata_w.iloc[:, 0].astype(np.int64) * 1_000_000_000 +
           ata_w.iloc[:, 1].astype(np.int64)) - ref_ns
mem_rel = (mem_w.iloc[:, 0].astype(np.int64) * 1_000_000_000 +
           mem_w.iloc[:, 1].astype(np.int64)) - ref_ns

w_ata = ata_w[(ata_rel >= 0) & (ata_rel < T_WINDOW_NS)]
w_mem = mem_w[(mem_rel >= 0) & (mem_rel < T_WINDOW_NS)]

text = log_lines_to_text(w_ata, w_mem)
print(f"Example window text (WannaCry, window 0):")
print(f"  Lines: {len(w_ata)} ATA write + {len(w_mem)} mem write")
print(f"  Text:  {text[:300]}...")

Example window text (WannaCry, window 0):
  Lines: 0 ATA write + 21 mem write
  Text:  no storage write activity . memory-write 2mb-page high-entropy . memory-write 2mb-page moderate-entropy . memory-write 2mb-page high-entropy . memory-write 2mb-page high-entropy . memory-write 2mb-page high-entropy . memory-write 2mb-page high-entropy . memory-write 2mb-page low-entropy . memory-wri...


In [3]:
def extract_text_windows(exec_dir):
    """
    Extract text descriptions for all 0.1s windows across T_d = 30s.
    Returns list of 300 text strings — one per window.
    """
    ata_w_path = exec_dir / "ata_write.csv"
    mem_w_path = exec_dir / "mem_write.csv"

    ata_w = pd.DataFrame()
    mem_w = pd.DataFrame()

    if ata_w_path.exists() and ata_w_path.stat().st_size > 0:
        try:
            ata_w = pd.read_csv(ata_w_path, header=None)
        except:
            pass

    if mem_w_path.exists() and mem_w_path.stat().st_size > 0:
        try:
            mem_w = pd.read_csv(mem_w_path, header=None)
        except:
            pass

    # Compute reference timestamp
    ref_ns = None
    if not ata_w.empty:
        t = int(ata_w.iloc[0, 0]) * 1_000_000_000 + int(ata_w.iloc[0, 1])
        ref_ns = t
    if not mem_w.empty:
        t = int(mem_w.iloc[0, 0]) * 1_000_000_000 + int(mem_w.iloc[0, 1])
        ref_ns = t if ref_ns is None else min(ref_ns, t)

    if ref_ns is None:
        return ["no activity"] * 300

    # Add relative timestamps
    if not ata_w.empty:
        ata_rel = (ata_w.iloc[:, 0].astype(np.int64) * 1_000_000_000 +
                   ata_w.iloc[:, 1].astype(np.int64)) - ref_ns
        ata_w = ata_w.copy()
        ata_w["rel_ns"] = ata_rel

    if not mem_w.empty:
        mem_rel = (mem_w.iloc[:, 0].astype(np.int64) * 1_000_000_000 +
                   mem_w.iloc[:, 1].astype(np.int64)) - ref_ns
        mem_w = mem_w.copy()
        mem_w["rel_ns"] = mem_rel

    windows = []
    t = 0
    while t <= T_D_NS - T_WINDOW_NS:
        t_end = t + T_WINDOW_NS

        w_ata = pd.DataFrame()
        w_mem = pd.DataFrame()

        if not ata_w.empty:
            mask = (ata_w["rel_ns"] >= t) & (ata_w["rel_ns"] < t_end)
            w_ata = ata_w[mask]
        if not mem_w.empty:
            mask = (mem_w["rel_ns"] >= t) & (mem_w["rel_ns"] < t_end)
            w_mem = mem_w[mask]

        windows.append(log_lines_to_text(w_ata, w_mem))
        t += T_WINDOW_NS

    return windows


# Quick test
windows = extract_text_windows(test_exec)
print(f"Windows extracted: {len(windows)}")
print(f"\nWindow 0 (early):  {windows[0][:150]}")
print(f"\nWindow 50 (mid):   {windows[50][:150]}")
print(f"\nWindow 150 (late): {windows[150][:150]}")

Windows extracted: 300

Window 0 (early):  no storage write activity . memory-write 2mb-page high-entropy . memory-write 2mb-page moderate-entropy . memory-write 2mb-page high-entropy . memory-

Window 50 (mid):   storage-write small-write low-entropy . memory-write 2mb-page high-entropy . memory-write mmio near-zero-entropy . memory-write mmio near-zero-entropy

Window 150 (late): storage-write small-write moderate-entropy . storage-write small-write moderate-entropy . storage-write small-write moderate-entropy . storage-write s


In [4]:
# This cell processes all execution directories
# and produces text windows for the sentence transformer

meta_df = pd.read_csv(FEATURES_DIR / "metadata.csv")

all_windows  = []   # list of lists — each inner list is 300 window texts
all_labels   = []   # class name per execution
all_3way     = []   # 3-way label per execution
all_groups   = []   # original / variant

print(f"Processing {len(meta_df)} execution directories...")
print("This may take a few minutes.\n")

for i, row in meta_df.iterrows():
    exec_dir   = Path(row["exec_dir"])
    label      = row["label"]
    group      = row["group"]
    label_3way = get_3way_label(label) if group == "original" else "variant"

    windows = extract_text_windows(exec_dir)

    all_windows.append(windows)
    all_labels.append(label)
    all_3way.append(label_3way)
    all_groups.append(group)

    if (i + 1) % 20 == 0:
        print(f"  Processed {i+1}/{len(meta_df)}")

all_labels = np.array(all_labels)
all_3way   = np.array(all_3way)
all_groups = np.array(all_groups)

print(f"\nDone.")
print(f"Total executions:  {len(all_windows)}")
print(f"Windows per exec:  {len(all_windows[0])}")
print(f"\n3-way label distribution:")
for label in np.unique(all_3way):
    print(f"  {label:<20}: {(all_3way == label).sum()}")

Processing 190 execution directories...
This may take a few minutes.

  Processed 20/190
  Processed 40/190
  Processed 60/190
  Processed 80/190
  Processed 100/190
  Processed 120/190
  Processed 140/190
  Processed 160/190
  Processed 180/190

Done.
Total executions:  190
Windows per exec:  300

3-way label distribution:
  benign              : 30
  legit_encryption    : 30
  ransomware          : 60
  variant             : 70


In [5]:
# Flatten all windows into one list for batch embedding
# Then reshape back to (n_executions, 300, 384)

print("Loading sentence transformer...")
st_model = SentenceTransformer("all-MiniLM-L6-v2")
emb_dim  = st_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {emb_dim}")

# Flatten
flat_windows = [w for exec_windows in all_windows for w in exec_windows]
print(f"\nTotal windows to embed: {len(flat_windows)}")
print("Embedding... (this will take several minutes)")

flat_embeddings = st_model.encode(
    flat_windows,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nFlat embeddings shape: {flat_embeddings.shape}")

# Reshape to (n_executions, 300, 384)
n_exec    = len(all_windows)
n_windows = len(all_windows[0])
text_seqs  = flat_embeddings.reshape(n_exec, n_windows, emb_dim)

print(f"Reshaped: {text_seqs.shape}  (executions, windows, embedding_dim)")

# Save to disk — this takes time to compute, don't redo it
np.save(FEATURES_DIR / "text_sequences.npy", text_seqs)
np.save(FEATURES_DIR / "text_labels.npy",    all_labels)
np.save(FEATURES_DIR / "text_3way.npy",      all_3way)
np.save(FEATURES_DIR / "text_groups.npy",    all_groups)
print("Saved to disk.")

Loading sentence transformer...


Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 4442.20it/s]
/tmp/ipykernel_30671/1733396111.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  emb_dim  = st_model.get_sentence_embedding_dimension()


Embedding dimension: 384

Total windows to embed: 57000
Embedding... (this will take several minutes)


Batches: 100%|████████████████████████████████| 223/223 [16:15<00:00,  4.37s/it]



Flat embeddings shape: (57000, 384)
Reshaped: (190, 300, 384)  (executions, windows, embedding_dim)
Saved to disk.


## Output

This notebook writes:
- `features/raw_text_windows.pkl` — raw text strings (n_exec × 300)
- `features/text_sequences.npy` — MiniLM embeddings (n_exec × 300 × 384)
- `features/labels.npy` — integer class labels (n_exec,)

Downstream notebooks load these cached arrays directly.